In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# 1. 데이터 로드
df = pd.read_csv('/content/sample_data/xgboost_data.csv')

# 피처 정의
model_a_features = [
    'price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
    'score_ch1', 'score_ch2', 'score_ch3', 'score_ch4', 'score_ch5'
]

model_b_features = {
    'sneakers': ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch3', 'score_ch4'],   # Granger SIGNIFICANT
    'cards':    ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch1'],                # Granger SIGNIFICANT
    'lego':     ['price_vs_ma3', 'price_chg_lag1', 'price_chg_lag2', 'price_chg_lag3',
                 'score_ch1'],                # Granger MARGINAL (p=0.054)
}

#초기 학습 데이터 부족으로 인한 모델 성능 저하를 막고,
#분석에서 밝혀낸 6개월의 가격 반영 주기를 맞추기 위해 test_size=6, n_splits=4로 설정하였다"
tscv = TimeSeriesSplit(n_splits=4, test_size=6)

# 모든 item_id에 대한 DM 검정 데이터를 저장할 리스트
all_dm_data = []

# --- 모든 item_id에 대해 DM 검정 데이터 추출 및 모델 학습/예측 ---
for selected_item_id in df['item_id'].unique():
    subset_for_dm = df[df['item_id'] == selected_item_id].copy()
    subset_for_dm = subset_for_dm.sort_values('year_month').reset_index(drop=True)
    asset_type_for_dm = subset_for_dm['asset_type'].iloc[0]

    X_a_dm = subset_for_dm[model_a_features]
    X_b_dm = subset_for_dm[model_b_features[asset_type_for_dm]]
    y_dm = subset_for_dm['mean_price']

    # TimeSeriesSplit의 마지막 폴드를 추출
    train_idx, test_idx = None, None
    for t_idx, te_idx in tscv.split(subset_for_dm):
        train_idx, test_idx = t_idx, te_idx

    if train_idx is None or test_idx is None or len(test_idx) == 0:
        print(f"Warning: item_id '{selected_item_id}' 에는 TimeSeriesSplit을 위한 데이터가 충분하지 않거나 테스트 폴드가 비어 있습니다. 이 아이템은 건너뜁니다.")
        continue

    # 마지막 폴드의 학습 데이터와 테스트 데이터 준비
    X_train_A_dm, X_test_A_dm = X_a_dm.iloc[train_idx], X_a_dm.iloc[test_idx]
    X_train_B_dm, X_test_B_dm = X_b_dm.iloc[train_idx], X_b_dm.iloc[test_idx]
    y_train_dm, y_test_dm = y_dm.iloc[train_idx], y_dm.iloc[test_idx]

    # --- 모델 학습 및 예측 ---
    model_A_dm = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05,
                                  random_state=42, eval_metric='rmse', verbosity=0)
    model_A_dm.fit(X_train_A_dm, y_train_dm)

    model_B_dm = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05,
                                  random_state=42, eval_metric='rmse', verbosity=0)
    model_B_dm.fit(X_train_B_dm, y_train_dm)

    # 예측값 추출
    y_actual = y_test_dm.values if hasattr(y_test_dm, 'values') else y_test_dm
    y_pred_A = model_A_dm.predict(X_test_A_dm)
    y_pred_B = model_B_dm.predict(X_test_B_dm)

    # DM 검정용 데이터프레임 생성 및 리스트에 추가
    dm_data_item = pd.DataFrame({
        'item_id': selected_item_id,
        'Actual': y_actual,
        'Model_A_Pred': y_pred_A,
        'Model_B_Pred': y_pred_B
    })
    all_dm_data.append(dm_data_item)

# 모든 아이템의 데이터를 하나로 합칩니다.
if all_dm_data:
    dm_data_all_items = pd.concat(all_dm_data, ignore_index=True)
    # CSV 파일로 저장
    dm_data_all_items.to_csv('dm_test_input_all_items.csv', index=False)
    print(f"\nDM 검정용 데이터 추출 완료: 'dm_test_input_all_items.csv' for all {len(df['item_id'].unique())} items.")
    print("\n--- DM 검정 데이터 샘플 (상위 5개 아이템) ---")
    print(dm_data_all_items.groupby('item_id').head(1))
else:
    print("DM 검정 데이터를 생성할 아이템이 없습니다. 데이터 및 TimeSeriesSplit 설정을 확인하세요.")


DM 검정용 데이터 추출 완료: 'dm_test_input_all_items.csv' for all 15 items.

--- DM 검정 데이터 샘플 (상위 5개 아이템) ---
             item_id   Actual  Model_A_Pred  Model_B_Pred
0   sneakers_jordan1   102.53    118.567375    170.784302
6     sneakers_panda    68.84     94.600166    112.192337
12    sneakers_yeezy   293.48    296.927673    325.969879
18   sneakers_travis  1229.71   1361.020020   1454.645630
24    sneakers_nb550   122.46    115.411057    116.721130
30  cards_charizard1   105.00    119.570702    128.595322
36  cards_charizard2   500.00    390.033539    399.298340
42     cards_umbreon  1429.47    673.530334    592.340332
48    cards_rayquaza   628.59    277.982269    319.294159
54     cards_pikachu   139.70    139.541443    127.904015
60       lego_falcon   609.39    731.183716    725.528442
66     lego_hogwarts   365.36    350.968414    359.716248
72      lego_titanic   554.11    588.199219    552.837646
78      lego_porsche   142.63    146.125748    144.448456
84      lego_bugatti   338.25

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import t

# Custom Diebold-Mariano Test Function
def custom_dm_test(errors_A, errors_B, h=1, power=2):
    # 1. Calculate loss differential
    loss_A = np.abs(errors_A)**power
    loss_B = np.abs(errors_B)**power
    d_t = loss_A - loss_B

    # 2. Calculate sample mean of d_t
    mean_d = np.mean(d_t)

    # 3. Calculate long-run variance using Newey-West estimator
    # This part handles autocorrelation up to lag h
    T = len(d_t)
    gamma_0 = np.mean((d_t - mean_d)**2)
    gamma_k_sum = 0.0

    for k in range(1, h + 1):
        # Autocovariance at lag k
        gamma_k = np.mean((d_t[k:] - mean_d) * (d_t[:-k] - mean_d))
        # Newey-West weight
        weight = 1 - (k / (h + 1))
        gamma_k_sum += weight * gamma_k

    long_run_variance = gamma_0 + 2 * gamma_k_sum

    # Ensure variance is not negative or zero
    if long_run_variance <= 0:
        # Fallback for insufficient data or zero variance (e.g., if d_t is constant)
        # Can return NaN or handle as an exception
        return np.nan, np.nan

    # 4. Calculate DM statistic
    dm_stat = mean_d / np.sqrt(long_run_variance / T)

    # 5. Calculate p-value (two-sided test with t-distribution for small samples)
    # Degrees of freedom T-1
    p_value = 2 * (1 - t.cdf(np.abs(dm_stat), df=T - 1))

    return dm_stat, p_value

# 1. 이전에 저장된 DM 검정용 데이터 파일 로드
dm_data_all_items = pd.read_csv('dm_test_input_all_items.csv')

# DM 검정 결과를 저장할 리스트
dm_test_results = []

# 2. 각 item_id 별로 DM 검정 수행
for item_id in dm_data_all_items['item_id'].unique():
    item_subset = dm_data_all_items[dm_data_all_items['item_id'] == item_id].copy()

    # 테스트 데이터가 충분하지 않은 경우 건너뛰기
    if len(item_subset) < 2: # DM 검정을 위해서는 최소 2개 이상의 데이터 포인트가 필요합니다.
        print(f"Warning: item_id '{item_id}' 에 대한 DM 검정 데이터가 너무 적습니다 ({len(item_subset)}개). 이 아이템은 건너뜁니다.")
        continue

    y_actual = item_subset['Actual']
    y_pred_A = item_subset['Model_A_Pred']
    y_pred_B = item_subset['Model_B_Pred']

    # 각 모델의 예측 오차 계산
    errors_A = y_actual - y_pred_A
    errors_B = y_actual - y_pred_B

    # Diebold-Mariano 검정 수행 (커스텀 함수 사용)
    try:
        dm_stat, p_value = custom_dm_test(errors_A, errors_B, h=1, power=2)

        # NaN 결과 처리
        if np.isnan(dm_stat) or np.isnan(p_value):
            winner = "DM 검정 불가 (데이터 부족 또는 분산 0)"
        else:
            # DM 검정 결과 해석
            # 귀무가설(H0): 두 모델의 예측 정확도에 통계적으로 유의미한 차이가 없다.
            # 대립가설(H1): 두 모델의 예측 정확도에 통계적으로 유의미한 차이가 있다.
            # dm_stat > 0 이면 Model B의 예측 오차가 평균적으로 Model A보다 작음 (Model B가 더 좋음)
            # dm_stat < 0 이면 Model A의 예측 오차가 평균적으로 Model B보다 작음 (Model A가 더 좋음)
            if p_value < 0.05: # 유의수준 5% 기준
                if dm_stat > 0:
                    winner = "Model B (우수)"
                else: # dm_stat < 0
                    winner = "Model A (우수)"
            else:
                winner = "유의미한 차이 없음"

        dm_test_results.append({
            'item_id': item_id,
            'DM_Statistic': dm_stat,
            'P_Value': p_value,
            'Winner': winner
        })
    except Exception as e:
        print(f"Warning: item_id '{item_id}' 에 대한 DM 검정 중 오류 발생: {e}. 해당 아이템은 건너뜁니다.")
        dm_test_results.append({
            'item_id': item_id,
            'DM_Statistic': np.nan,
            'P_Value': np.nan,
            'Winner': f"Error: {e}"
        })

# 결과를 DataFrame으로 변환하여 출력
dm_results_df = pd.DataFrame(dm_test_results)

print("\n--- Diebold-Mariano Test Results (DM 검정 결과) ---")
print(dm_results_df.to_string(index=False))



--- Diebold-Mariano Test Results (DM 검정 결과) ---
         item_id  DM_Statistic  P_Value       Winner
sneakers_jordan1     -0.743247 0.490758   유의미한 차이 없음
  sneakers_panda      0.685007 0.523797   유의미한 차이 없음
  sneakers_yeezy     -3.791586 0.012738 Model A (우수)
 sneakers_travis     -1.180970 0.290725   유의미한 차이 없음
  sneakers_nb550     -0.328649 0.755739   유의미한 차이 없음
cards_charizard1     -1.931679 0.111251   유의미한 차이 없음
cards_charizard2      0.938956 0.390861   유의미한 차이 없음
   cards_umbreon     -0.214075 0.838945   유의미한 차이 없음
  cards_rayquaza     -0.173256 0.869245   유의미한 차이 없음
   cards_pikachu     -1.045421 0.343712   유의미한 차이 없음
     lego_falcon      0.351777 0.739345   유의미한 차이 없음
   lego_hogwarts      1.037628 0.346995   유의미한 차이 없음
    lego_titanic     -0.547976 0.607274   유의미한 차이 없음
    lego_porsche     -1.806477 0.130664   유의미한 차이 없음
    lego_bugatti     -0.445634 0.674495   유의미한 차이 없음
